# 03 — Feature selection, logistic regression and scorecard

**Goal:** pick a small, stable, explainable set of features; fit a logistic regression on
their WoE values; turn it into a points-based scorecard; compare with a black-box challenger.

In [ ]:
import os, sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pdmodel import config as C
pd.set_option("display.float_format", "{:.4f}".format)
PROC = ROOT / "data" / "processed"
from pdmodel import model
from pdmodel.validation import discrimination

In [ ]:
S = pd.read_pickle(PROC / "samples.pkl")
binner = pd.read_pickle(PROC / "binner.pkl")
Xw = {k: binner.transform(v) for k, v in S.items()}
y_train = S["train"]["default"]

## 1. Filter by IV and correlation
Walk down the IV ranking; skip weak (IV < 0.02) or suspicious (IV > 0.5) features, and skip
any feature whose WoE is > 0.7 correlated with one already kept.

In [ ]:
shortlist, sel_log = model.select_by_iv_and_correlation(binner, Xw["train"])
sel_log

## 2. Backward elimination
Refit and drop features until all coefficients are **significant** and **negative**.
(Negative because higher WoE = safer = lower PD. A positive sign means collinearity
is distorting the model.)

In [ ]:
final, elim_log = model.backward_eliminate(Xw["train"], y_train, shortlist)
print("final features:", final)
elim_log

In [ ]:
res = model.fit_logit(Xw["train"][final], y_train)
print(res.summary2().tables[0].to_string(header=False))
model.coefficient_table(res, Xw["train"][final])

## 3. Scorecard scaling
`score = offset + factor · ln(good:bad odds)`, with `factor = PDO / ln 2`.
With the defaults, **600 points = 30:1 odds, and +20 points doubles the odds**.

In [ ]:
card = model.Scorecard(binner, final, res)
print(f"factor = {card.factor:.2f}, offset = {card.offset:.2f}")
pts = card.points_table()
pts

Sanity check: summing the points for one applicant should reproduce `card.score()` (up to rounding).

In [ ]:
one = S["test"].iloc[[0]]
manual = sum(pts.loc[(pts.feature == f) & (pts.bin == binner.bins[f].bin_labels(one[f]).iloc[0]), "points"].iloc[0]
             for f in final)
print(f"points table: {manual}   score(): {card.score(one)[0]:.1f}")

## 4. Challenger: gradient boosting
How much predictive power does the simple scorecard give up?

In [ ]:
gbm = model.fit_challenger(S["train"])
pd.DataFrame({k: {"scorecard_gini": discrimination(v["default"], card.predict_pd(v))["gini"],
                  "challenger_gini": discrimination(v["default"], model.challenger_pd(gbm, v))["gini"]}
              for k, v in S.items()}).T

In [ ]:
pd.to_pickle({"features": final, "logit": res, "scorecard": card, "challenger": gbm}, PROC / "model.pkl")

## ✍️ Interview prep
1. Why would a bank pick the scorecard even if the challenger has a higher Gini?
2. What does the PDO parameter mean in plain English?
3. How would you explain the points table to a credit officer who doesn't know statistics?
4. What's VIF and why check it?